# ICT Multi-Symbol Backtest — S-006 M3

**Purpose:** Fetch real 2026 OHLCV data for all pairs in `data/ict_validate_manifest.csv`,
run the ICT backtest harness across every pair, then produce a go/no-go validation report.

**Inputs required:**
- Colab secret `GITHUB_PAT` — read-only PAT to clone the repo.
- (Optional) Colab secret `BYBIT_API_KEY` / `BYBIT_API_SECRET` — only needed if
  fetching from Bybit instead of Binance public API.

**Outputs saved to Drive:**
- `MyDrive/ict-bot-research/backtest-runs/ict_multi_YYYYMMDD.json` — raw backtest report
- `MyDrive/ict-bot-research/backtest-runs/ict_validation_report_YYYYMMDD.md` — go/no-go report

**Go criteria (M7 Phase 3):** ≥ 50 trades total · WR ≥ 55 % · avg R > 0

Run cells top-to-bottom. Each cell is idempotent.

In [ ]:
# Cell 1 — Install dependencies
!pip install -q ccxt yfinance pandas

In [ ]:
# Cell 2 — Mount Drive and define output paths
from google.colab import drive
drive.mount('/content/drive')

import os
from datetime import datetime

RUN_DATE   = datetime.utcnow().strftime('%Y%m%d')
DRIVE_DIR  = '/content/drive/MyDrive/ict-bot-research/backtest-runs'
os.makedirs(DRIVE_DIR, exist_ok=True)

JSON_OUT   = f'{DRIVE_DIR}/ict_multi_{RUN_DATE}.json'
REPORT_OUT = f'{DRIVE_DIR}/ict_validation_report_{RUN_DATE}.md'
print(f'Outputs → {JSON_OUT}')
print(f'         {REPORT_OUT}')

In [ ]:
# Cell 3 — Clone / update repo
from google.colab import userdata
from getpass import getpass

try:
    PAT = userdata.get('GITHUB_PAT')
except Exception:
    PAT = getpass('GitHub PAT (read-only): ')

REPO = 'the-lizardking/ict-trading-bot'
REPO_DIR = '/content/ict-trading-bot'

if not os.path.exists(REPO_DIR):
    !git clone -q https://$PAT@github.com/{REPO}.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull -q origin main

%cd {REPO_DIR}
!git log --oneline -3

In [ ]:
# Cell 4 — Fetch OHLCV data for each manifest pair
#
# Crypto pairs  → Binance public REST via ccxt (no API key needed)
# Equity pairs  → yfinance
#
# Data is written directly to the paths listed in the manifest so that
# backtest_ict.py --manifest can read them without any path remapping.

import ccxt
import yfinance as yf
import pandas as pd
import csv, os
from pathlib import Path

MANIFEST = pd.read_csv('data/ict_validate_manifest.csv')
print(MANIFEST.to_string(index=False))

# Date range: full 2026 year-to-date
START = '2026-01-01'
END   = datetime.utcnow().strftime('%Y-%m-%d')

# ccxt timeframe map
TF_CCXT = {'5m': '5m', '15m': '15m', '1h': '1h'}
# yfinance interval map
TF_YF   = {'5m': '5m', '15m': '15m', '1h': '1h'}

CRYPTO_SYMBOLS = {'BTCUSDT', 'ETHUSDT'}
EQUITY_SYMBOLS = {'SPY', 'QQQ'}

binance = ccxt.binance({'enableRateLimit': True})

def fetch_crypto(symbol, timeframe, out_path):
    """Fetch via Binance public OHLCV endpoint and save to CSV."""
    tf = TF_CCXT[timeframe]
    since = binance.parse8601(f'{START}T00:00:00Z')
    all_bars = []
    while True:
        bars = binance.fetch_ohlcv(symbol, tf, since=since, limit=1000)
        if not bars:
            break
        all_bars.extend(bars)
        since = bars[-1][0] + 1
        if len(bars) < 1000:
            break
    df = pd.DataFrame(all_bars, columns=['ts', 'open', 'high', 'low', 'close', 'volume'])
    df['timestamp'] = pd.to_datetime(df['ts'], unit='ms', utc=True).dt.tz_localize(None)
    df = df[['timestamp', 'open', 'high', 'low', 'close', 'volume']]
    Path(out_path).parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(out_path, index=False)
    print(f'  {symbol} {timeframe}: {len(df)} rows → {out_path}')

def fetch_equity(symbol, timeframe, out_path):
    """Fetch via yfinance and save to CSV."""
    tf = TF_YF[timeframe]
    ticker = yf.Ticker(symbol)
    df = ticker.history(start=START, end=END, interval=tf, auto_adjust=True)
    df = df.reset_index().rename(columns={
        'Datetime': 'timestamp', 'Date': 'timestamp',
        'Open': 'open', 'High': 'high', 'Low': 'low',
        'Close': 'close', 'Volume': 'volume',
    })
    df['timestamp'] = pd.to_datetime(df['timestamp']).dt.tz_localize(None)
    df = df[['timestamp', 'open', 'high', 'low', 'close', 'volume']]
    Path(out_path).parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(out_path, index=False)
    print(f'  {symbol} {timeframe}: {len(df)} rows → {out_path}')

print('\nFetching OHLCV data...')
for _, row in MANIFEST.iterrows():
    sym, tf, path = row['symbol'], row['timeframe'], row['path']
    if sym in CRYPTO_SYMBOLS:
        fetch_crypto(sym, tf, path)
    elif sym in EQUITY_SYMBOLS:
        fetch_equity(sym, tf, path)
    else:
        print(f'  WARNING: unknown symbol {sym}, skipping')
print('Done.')

In [ ]:
# Cell 5 — Run the ICT backtest across all manifest pairs
import subprocess, sys

result = subprocess.run(
    [sys.executable, 'bin/backtest_ict.py',
     '--manifest', 'data/ict_validate_manifest.csv',
     '--output',   JSON_OUT,
     '--quiet'],
    capture_output=True, text=True,
    env={**os.environ, 'PYTHONPATH': REPO_DIR},
)
print(result.stdout)
if result.returncode not in (0, 1):   # 1 = some pairs failed, still ok
    print('STDERR:', result.stderr)
    raise RuntimeError(f'backtest_ict.py exited {result.returncode}')
print(f'Report written to {JSON_OUT}')

In [ ]:
# Cell 6 — Run the analyzer and produce the go/no-go validation report
result = subprocess.run(
    [sys.executable, 'bin/analyze_ict_results.py',
     '--input',      JSON_OUT,
     '--output',     REPORT_OUT,
     '--min-trades', '50',
     '--min-wr',     '55'],
    capture_output=True, text=True,
    env={**os.environ, 'PYTHONPATH': REPO_DIR},
)
print(result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr)
    raise RuntimeError(f'analyze_ict_results.py exited {result.returncode}')

In [ ]:
# Cell 7 — Display the validation report inline
from IPython.display import Markdown, display
display(Markdown(open(REPORT_OUT).read()))

In [ ]:
# Cell 8 — (Optional) Copy validation report into the repo and commit
#
# Run this cell only if the go/no-go report is ready to be checked in.
# Requires a PAT with write access (push permission).

import shutil

REPO_REPORT = f'{REPO_DIR}/docs/sprint-plans/ict-validation-report.md'
shutil.copy(REPORT_OUT, REPO_REPORT)

!git -C {REPO_DIR} config user.email 'colab@ict-bot'
!git -C {REPO_DIR} config user.name  'Colab'
!git -C {REPO_DIR} add docs/sprint-plans/ict-validation-report.md
!git -C {REPO_DIR} commit -m 'feat(s006-m3): ICT multi-symbol validation report {RUN_DATE}'
!git -C {REPO_DIR} push https://$PAT@github.com/{REPO}.git main
print('Committed and pushed.')

## Handoff to Claude

After running all cells, copy the following back into the Claude Code session:

1. The printed **Verdict** line from Cell 6 (`GO` / `NO-GO` + trade count + WR + avg R).
2. The full contents of `REPORT_OUT` (displayed in Cell 7), or the Drive path so Claude can read it.
3. Any error output from Cells 4–6.

Claude will then:
- If **GO**: open a PR to wire the ICT strategy into the live pipeline (S-006 M4).
- If **NO-GO**: document the shortfall and recommend next data-gathering steps.